# Task 4 : Fine Tuning BERT on Kaggle Dataset

## 1. Loading all important libraries

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np
import evaluate

## 2. Loading the Dataset
### Dataset loading from Hugging Face library to faster import and faster model training and performance

In [5]:
dataset = load_dataset("imdb")

In [49]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

## Data Preprocessing and Feature Analysis - No missing values found

In [51]:
print(dataset["train"][:5])

{'text': ['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far b

In [55]:
print(dataset["train"][0])   # first row

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [57]:
print(dataset["train"][1])   # second row

{'text': '"I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn\'t matter what one\'s political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn\'t true. I\'ve seen R-rated films with male nudity. Granted, they only offer some fleeting views, but where are the R-rated films with gaping vulvas and flapping labia? Nowhere, because they don\'t exist. The same goes for those crappy cable shows: schlongs swinging in the breeze but not a clitoris in sight. And those pretentious indie movies like The Brown Bunny, in which we\'re treated to the site of Vincent Gallo\'s throbbing johnson, but not a trace of pink visible on Chloe Sevigny. Before crying (or implying) "double-standard" in matters of nudity, the mentally obtuse should take into account one unavoidably obvious anatomical difference between men and women: there are no genitals on display when actresses appears nude, 

In [63]:
print(dataset["train"].unique("label"))

[0, 1]


In [65]:
print(dataset["train"].features["label"].names)

['neg', 'pos']


In [67]:
# To count how many samples per label
from collections import Counter

print(Counter(dataset["train"]["label"]))

Counter({0: 12500, 1: 12500})


In [61]:
from pprint import pprint

pprint(dataset["train"][:1])

{'label': [0],
 'text': ['I rented I AM CURIOUS-YELLOW from my video store because of all the '
          'controversy that surrounded it when it was first released in 1967. '
          'I also heard that at first it was seized by U.S. customs if it ever '
          'tried to enter this country, therefore being a fan of films '
          'considered "controversial" I really had to see this for myself.<br '
          '/><br />The plot is centered around a young Swedish drama student '
          'named Lena who wants to learn everything she can about life. In '
          'particular she wants to focus her attentions to making some sort of '
          'documentary on what the average Swede thought about certain '
          'political issues such as the Vietnam War and race issues in the '
          'United States. In between asking politicians and ordinary denizens '
          'of Stockholm about their opinions on politics, she has sex with her '
          'drama teacher, classmates, and 

## Data Splitting

In [6]:
# Reduce size for FAST training
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(1000))

## 3. Loading tokenizer

In [8]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

## 4. Building Tokenization function

In [10]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True)

In [11]:
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

In [12]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 5. Loading the model

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | Details
------------------------+------------+--------
vocab_transform.weight  | UNEXPECTED |        
vocab_transform.bias    | UNEXPECTED |        
vocab_layer_norm.bias   | UNEXPECTED |        
vocab_projector.bias    | UNEXPECTED |        
vocab_layer_norm.weight | UNEXPECTED |        
pre_classifier.weight   | MISSING    |        
classifier.bias         | MISSING    |        
pre_classifier.bias     | MISSING    |        
classifier.weight       | MISSING    |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 6. Building the accuracy metrics function

In [16]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

## 7. Training Arguments

In [18]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,   # VERY FAST
    logging_steps=50,
    report_to="none"
)

## 8. Building the Trainer

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

## 9. Train the actual model

In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.274251,0.299201,0.880000


TrainOutput(global_step=250, training_loss=0.3996177749633789, metrics={'train_runtime': 3295.7795, 'train_samples_per_second': 0.607, 'train_steps_per_second': 0.076, 'total_flos': 244646587286592.0, 'train_loss': 0.3996177749633789, 'epoch': 1.0})

## 10. Evaluate

In [45]:
from sklearn.metrics import classification_report

# Faster evaluation dataset
eval_dataset = test_dataset.select(range(200))

# Get predictions
predictions = trainer.predict(eval_dataset)

y_pred = predictions.predictions.argmax(axis=1)
y_true = predictions.label_ids

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))


Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.88      0.87       104
           1       0.86      0.85      0.86        96

    accuracy                           0.86       200
   macro avg       0.86      0.86      0.86       200
weighted avg       0.86      0.86      0.86       200



## 11. Building continuous user flow

In [69]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    outputs = model(**inputs)
    logits = outputs.logits
    
    predicted_class_id = logits.argmax().item()
    
    if predicted_class_id == 1:
        return "Thankyou ! , For such a positive review ☺️"
    else:
        return "Felt bad for your comment but thankyou for pointing out the mistake through this negative review 😔"

while True:
    text = input("\nEnter a review (or type 'exit'): ")
    
    if text.lower() == "exit":
        break
    
    result = predict(text)
    print("Prediction:", result)


Enter a review (or type 'exit'):  Dhurandar 2 was one of the best movie i watched recently


Prediction: Thankyou ! , For such a positive review ☺️



Enter a review (or type 'exit'):  Project Hail Mary is the best theatre experience and I have watched it in this IMAX pure cinema and best vfx


Prediction: Thankyou ! , For such a positive review ☺️



Enter a review (or type 'exit'):  The theatre ambience was good and the overall movies are also good


...


Enter a review (or type 'exit'):  I watched every marvel movie but some marvel movies are unambigous to watch and to rate them


Prediction: Felt bad for your comment but thankyou for pointing out the mistake through this negative review 😔



Enter a review (or type 'exit'):  overall movie experience was not good or not upto the mark


Prediction: Felt bad for your comment but thankyou for pointing out the mistake through this negative review 😔



Enter a review (or type 'exit'):  exit
